# Chapter 13 - Nonlinear Twiss

With Truncated Power Series Algebra (TPSA) tools, Twiss parameters in SciBmad can be expressed as Taylor series in selected variables. In this chapter, the selected variable is mainly the momentum deviation $\delta$.

For example, a beta function can be represented as

$$
\beta(s,\delta) = \beta_0(s) + \frac{\partial \beta}{\partial \delta}(s)\delta + \frac{1}{2!}\frac{\partial^2 \beta}{\partial \delta^2}(s)\delta^2 + \cdots.
$$

The Taylor coefficients can then be read directly from the TPSA objects returned by `twiss`. This is different from a finite-difference scan: the derivatives are part of the map-based calculation itself.

We begin by loading the ESR lattice used in the local tutorial files.

In [ ]:
using SciBmad
using CairoMakie
using LaTeXStrings

set_theme!(theme_latexfonts())

In [ ]:
# The ESR lattice file lives one directory above this notebook.
# Current Beamlines versions use PhaseReference; this lattice file uses PhaseRef.
if !isdefined(Main, :PhaseRef) && isdefined(Main, :PhaseReference)
    const PhaseRef = PhaseReference
end

ring = include("../esr-main-18GeV-1IP.jl")

## 13.1 Initializing Twiss

A standard `twiss` call returns the coupled Twiss functions around the ring. The table columns `beta_1` and `beta_2` are the two transverse normal-mode beta functions in the Teng-Edwards coupling formalism.

In [ ]:
tw = twiss(ring)
t = tw.table

In [ ]:
fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Beta Function [m]")

lines!(ax, t.s, t.beta_1, label=L"\beta_1")
lines!(ax, t.s, t.beta_2, label=L"\beta_2")
axislegend(ax, position=:rt)

fig

## 13.2 TPSA Operations

The closed orbit returned by `twiss` is also a TPSA object. At a fixed location, it can be interpreted as a Taylor series in the phase-space variables. For this lattice, the last coordinate is the momentum deviation $\delta$.

$$
x_{co}(\delta) = x_0 + \eta_x\delta + \frac{1}{2!}\frac{\partial \eta_x}{\partial \delta}\delta^2 + \cdots.
$$

The monomial index `[0, 0, 0, 0, 0, 1]` extracts the coefficient of $\delta$. Therefore, it gives the periodic dispersion $\eta_x$.

In [ ]:
# Inspect the closed orbit at the beginning of the ring.
t.orbit_x[1]

In [ ]:
eta_x = map(x -> x[[0, 0, 0, 0, 0, 1]], t.orbit_x)
eta_y = map(x -> x[[0, 0, 0, 0, 0, 1]], t.orbit_y)

fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Dispersion [m]")

lines!(ax, t.s, eta_x, label=L"\eta_x")
lines!(ax, t.s, eta_y, label=L"\eta_y")
axislegend(ax, position=:rt)

fig

To obtain second-order quantities, use a higher-order TPSA descriptor. `Descriptor(6, 2)` means six variables, truncated at second order.

In [ ]:
D2 = Descriptor(6, 2)
tw2 = twiss(ring; GTPSA_descriptor=D2)
t2 = tw2.table

In [ ]:
# The coefficient of delta^2 contains the Taylor factor 1/2!.
deta_ddelta_x = map(x -> 2 * x[[0, 0, 0, 0, 0, 2]], t2.orbit_x)
deta_ddelta_y = map(x -> 2 * x[[0, 0, 0, 0, 0, 2]], t2.orbit_y)

fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Second-Order Dispersion [m]")

lines!(ax, t2.s, deta_ddelta_x, label=L"\frac{\partial \eta_x}{\partial \delta}")
lines!(ax, t2.s, deta_ddelta_y, label=L"\frac{\partial \eta_y}{\partial \delta}")
axislegend(ax, position=:rt)

fig

The same coefficient extraction works for the beta functions. Here we extract the design beta, the first derivative with respect to $\delta$, and the second derivative with respect to $\delta$.

In [ ]:
beta_1 = map(x -> x[[0, 0, 0, 0, 0, 0]], t2.beta_1)
beta_2 = map(x -> x[[0, 0, 0, 0, 0, 0]], t2.beta_2)

dbeta_ddelta_1 = map(x -> x[[0, 0, 0, 0, 0, 1]], t2.beta_1)
dbeta_ddelta_2 = map(x -> x[[0, 0, 0, 0, 0, 1]], t2.beta_2)

d2beta_ddelta2_1 = map(x -> 2 * x[[0, 0, 0, 0, 0, 2]], t2.beta_1)
d2beta_ddelta2_2 = map(x -> 2 * x[[0, 0, 0, 0, 0, 2]], t2.beta_2)

In [ ]:
fig = Figure(fontsize=22, size=(900, 650))

ax1 = Axis(fig[1, 1], xlabel="s [m]", ylabel=L"\partial\beta/\partial\delta \; [m]")
lines!(ax1, t2.s, dbeta_ddelta_1, label=L"\beta_1")
lines!(ax1, t2.s, dbeta_ddelta_2, label=L"\beta_2")
axislegend(ax1, position=:rt)

ax2 = Axis(fig[2, 1], xlabel="s [m]", ylabel=L"\partial^2\beta/\partial\delta^2 \; [m]")
lines!(ax2, t2.s, d2beta_ddelta2_1, label=L"\beta_1")
lines!(ax2, t2.s, d2beta_ddelta2_2, label=L"\beta_2")
axislegend(ax2, position=:rt)

fig

## 13.3 Examples: Chromatic Beats and Montague Functions

A common way to show the chromatic distortion of the beta function is the normalized beta derivative

$$
\frac{1}{\beta}\frac{\partial \beta}{\partial \delta}.
$$

This is often called chromatic beta-beat.

In [ ]:
fig = Figure(fontsize=22, size=(820, 500))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Chromatic Beta-Beat")

lines!(ax, t2.s, dbeta_ddelta_1 ./ beta_1, label=L"\frac{\partial\beta_1/\partial\delta}{\beta_1}")
lines!(ax, t2.s, dbeta_ddelta_2 ./ beta_2, label=L"\frac{\partial\beta_2/\partial\delta}{\beta_2}")
axislegend(ax, position=:rt)

fig

The Montague functions combine the chromatic distortion of both $\alpha$ and $\beta$:

$$
W = \sqrt{A^2 + B^2},
$$

$$
A = \frac{\partial\alpha}{\partial\delta} - \frac{\alpha}{\beta}\frac{\partial\beta}{\partial\delta}, \qquad
B = \frac{1}{\beta}\frac{\partial\beta}{\partial\delta}.
$$

In achromatic regions, these functions are useful phase-invariant diagnostics of chromatic optical distortion.

In [ ]:
alpha_1 = map(x -> x[[0, 0, 0, 0, 0, 0]], t2.alpha_1)
alpha_2 = map(x -> x[[0, 0, 0, 0, 0, 0]], t2.alpha_2)

dalpha_ddelta_1 = map(x -> x[[0, 0, 0, 0, 0, 1]], t2.alpha_1)
dalpha_ddelta_2 = map(x -> x[[0, 0, 0, 0, 0, 1]], t2.alpha_2)

W_1 = @. sqrt((dalpha_ddelta_1 - alpha_1 / beta_1 * dbeta_ddelta_1)^2 + (dbeta_ddelta_1 / beta_1)^2)
W_2 = @. sqrt((dalpha_ddelta_2 - alpha_2 / beta_2 * dbeta_ddelta_2)^2 + (dbeta_ddelta_2 / beta_2)^2)

In [ ]:
fig = Figure(fontsize=22, size=(700, 360))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Montague Functions")

lines!(ax, t2.s, W_1, label=L"W_1")
lines!(ax, t2.s, W_2, label=L"W_2")
axislegend(ax, position=:lt)

fig

The same idea can be extended to higher order. For the second-order Montague functions below, the coefficient of $\delta^2$ is used directly because the definition includes the Taylor factor $1/2!$.

In [ ]:
D3 = Descriptor(6, 3)
tw3 = twiss(ring; GTPSA_descriptor=D3)
t3 = tw3.table

beta_1_d3 = map(x -> x[[0, 0, 0, 0, 0, 0]], t3.beta_1)
beta_2_d3 = map(x -> x[[0, 0, 0, 0, 0, 0]], t3.beta_2)
alpha_1_d3 = map(x -> x[[0, 0, 0, 0, 0, 0]], t3.alpha_1)
alpha_2_d3 = map(x -> x[[0, 0, 0, 0, 0, 0]], t3.alpha_2)

d2alpha_coeff_1 = map(x -> x[[0, 0, 0, 0, 0, 2]], t3.alpha_1)
d2alpha_coeff_2 = map(x -> x[[0, 0, 0, 0, 0, 2]], t3.alpha_2)
d2beta_coeff_1 = map(x -> x[[0, 0, 0, 0, 0, 2]], t3.beta_1)
d2beta_coeff_2 = map(x -> x[[0, 0, 0, 0, 0, 2]], t3.beta_2)

W2_1 = @. sqrt((d2alpha_coeff_1 - alpha_1_d3 / beta_1_d3 * d2beta_coeff_1)^2 + (d2beta_coeff_1 / beta_1_d3)^2)
W2_2 = @. sqrt((d2alpha_coeff_2 - alpha_2_d3 / beta_2_d3 * d2beta_coeff_2)^2 + (d2beta_coeff_2 / beta_2_d3)^2)

In [ ]:
fig = Figure(fontsize=22, size=(760, 380))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Second-Order Montague Functions")

lines!(ax, t3.s, W2_1, label=L"W^{(2)}_1")
lines!(ax, t3.s, W2_2, label=L"W^{(2)}_2")
axislegend(ax, position=:lt)

fig

## 13.4 Analyzing Spin

SciBmad can also include spin in the nonlinear Twiss calculation. Setting `spin=true` adds the stable spin direction $\hat n$ to the Twiss table. The derivative

$$
\frac{\partial \hat n}{\partial \delta}
$$

measures how sensitively the spin direction changes with momentum deviation.

In [ ]:
tw_spin = twiss(ring; spin=true)
ts = tw_spin.table

In [ ]:
dn_ddelta_x = -map(x -> x[6], ts.n_x)
dn_ddelta_y = -map(x -> x[6], ts.n_y)
dn_ddelta_z = -map(x -> x[6], ts.n_z)
dn_ddelta_amp = sqrt.(vec(sum(abs2, [dn_ddelta_x dn_ddelta_y dn_ddelta_z], dims=2)))

fig = Figure(fontsize=22, size=(760, 380))
ax = Axis(fig[1, 1], xlabel="s [m]", ylabel="Spin-Orbit Coupling")

lines!(ax, ts.s, dn_ddelta_x, label=L"\frac{\partial n_x}{\partial\delta}")
lines!(ax, ts.s, dn_ddelta_y, label=L"\frac{\partial n_y}{\partial\delta}")
lines!(ax, ts.s, dn_ddelta_z, label=L"\frac{\partial n_z}{\partial\delta}")
lines!(ax, ts.s, dn_ddelta_amp, label="|dn/ddelta|")
Legend(fig[1, 2], ax)

fig

## 13.5 Exercises

Exercises will be added after we decide which skills should be emphasized in this chapter.